# importing libraries

In [1]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.1-py3-none-any.whl.metadata (10.0 kB)
Using cached pybind11-3.0.1-py3-none-any.whl (293 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4498213 sha256=72f3796a3d353136073dc3fc6c41170d03216d78cc7f506b1e814a8845680aea
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, f1_score
import fasttext

# loading dataset

In [3]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# preprocessing

In [4]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
df.rename(columns={"v1": "label", "v2": "text"}, inplace=True)
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train = train_test_split(df['text'], df['label_enc'], test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = X_train.to_numpy(), X_test.to_numpy(), y_train.to_numpy(), y_test.to_numpy()

In [5]:
with open('training_data.txt', 'w') as f:
    for sentence, target in zip(X_train, y_train):
        # FastText expects labels as separate tokens, usually at the start
        clean_sentence = str(sentence).lower().replace('\n', ' ')
        f.write(f"__label__{target} {clean_sentence}\n")

In [6]:
with open('test_data.txt', 'w') as f:
    for sentence, target in zip(X_test, y_test):
        # FastText expects labels as separate tokens, usually at the start
        clean_sentence = str(sentence).lower().replace('\n', ' ')
        f.write(f"__label__{target} {clean_sentence}\n")

# model training

In [7]:
model = fasttext.train_supervised('training_data.txt')
# model = fasttext.train_supervised(input='training_data.txt', lr=0.01, loss='softmax', epoch=5)

In [8]:
def print_results(N, p, r):
    print("N\t" + str(N)) # test set size
    print("P@{}  {:.3f}".format(1, p)) # precision
    print("R@{}  {:.3f}".format(1, r)) # recall
    print(f"F1   {(p * r * 2 / (p + r)):.3f}");

# This should now show proper counts and metrics
print_results(*model.test('test_data.txt'))

N	1115
P@1  0.982
R@1  0.982
F1   0.982


# model evaluation

In [13]:
test_messages = [
    "Hey, are we meeting today?",
    "WIN cash now!!! Click the link",
    "Your OTP is 348921",
    "You have won $1000!",
    "Congratulations! You have won a free lottery ticket",
    "Don't forget to submit the assignment",
    "URGENT! You won a cash prize. Call immediately!",
    "My friend won a prize yesterday"
]

y_pred = model.predict(test_messages)
for i in range(len(test_messages)):
    print(f"{'Ham' if y_pred[0][i][0]=='__label__0' else 'Spam'} → {test_messages[i]}")
    # print(f"{y_pred[0][i][0]} → {test_messages[i]}")

Ham → Hey, are we meeting today?
Ham → WIN cash now!!! Click the link
Ham → Your OTP is 348921
Ham → You have won $1000!
Spam → Congratulations! You have won a free lottery ticket
Ham → Don't forget to submit the assignment
Spam → URGENT! You won a cash prize. Call immediately!
Ham → My friend won a prize yesterday


In [40]:
X_test2 = list()
for i in X_test:
    X_test2.append(str(i).lower().replace('\n', ' '))

y_pred = model.predict(X_test2)
y_pred2 = list()

for i in range(len(y_pred[0])):
    y_pred2.append(int(y_pred[0][i][0][-1]))

In [42]:
print('\ny_test')
print(y_test)

print('\ny_test.shape')
print(y_test.shape)

print('\ny_pred2')
print(y_pred2)

y_pred2 = np.array(y_pred2)
print('\ny_pred2')
print(y_pred2)

print('\ny_pred2.shape')
print(y_pred2.shape)


y_test
[0 0 0 ... 0 0 0]

y_test.shape
(1115,)

y_pred2
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 

In [44]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred2))
print(f1_score(y_test, y_pred2))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       966
           1       0.98      0.89      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

0.9295774647887324
